In [1]:
import pandas as pd
df=pd.read_csv("Companies_list_query.csv")

In [7]:
df["company_url"] = df["company"]
df["company"] = df["companyLabel"]
df["company_id"] = df["companyId"].astype(str).str.strip()
df["exchange"] = df["exchangeLabel"]



df["market_cap"] = pd.to_numeric(df["marketCap"], errors="coerce")
df["market_cap_date"] = pd.to_datetime(df["marketCapDate"], errors="coerce")

df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["revenue_date"] = pd.to_datetime(df["revenueDate"], errors="coerce")



for col in ["company", "company_id", "exchange", "ticker", "industries", "company_url"]:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})



df = df.sort_values(
    ["revenue", "company"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)



def first_non_null(series):
    series = series.dropna()
    return series.iloc[0] if len(series) > 0 else pd.NA


def join_unique(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]
    values = values[values.str.lower() != "nan"]
    return "; ".join(sorted(set(values)))




df_clean = (
    df.groupby("company_id", as_index=False)
      .agg({
          "company": first_non_null,
          "exchange": join_unique,
          "ticker": join_unique,
          "industries": join_unique,
          "market_cap": first_non_null,
          "market_cap_date": first_non_null,
          "company_url": first_non_null,
          "revenue": first_non_null,
          "revenue_date": first_non_null,
      })
)



df_clean["primary_ticker"] = (
    df_clean["ticker"]
    .fillna("")
    .str.split(";")
    .str[0]
    .str.strip()
    .replace("", pd.NA)
)


df_clean["has_real_company_name"] = (
    df_clean["company"].notna()
    & ~df_clean["company"].str.match(r"^Q\d+$", na=False)
)


print("Raw rows:", len(df))
print("Unique companies:", len(df_clean))
print("Companies with revenue:", df_clean["revenue"].notna().sum())
print("Companies with Wikidata market cap:", df_clean["market_cap"].notna().sum())
print("Companies with industry:", df_clean["industries"].notna().sum())
print("Companies with real company names:", df_clean["has_real_company_name"].sum())

df_clean.head(20)

Raw rows: 4268
Unique companies: 4190
Companies with revenue: 606
Companies with Wikidata market cap: 215
Companies with industry: 4190
Companies with real company names: 4185


,company_id,company,exchange,ticker,industries,market_cap,market_cap_date,company_url,revenue,revenue_date,primary_ticker,has_real_company_name
0,Q1001788,Buenaventura,New York Stock Exchange,BVN,mining,<NA>,<NA>,Buenaventura,<NA>,<NA>,BVN,True
1,Q1002992,Build-A-Bear Workshop,New York Stock Exchange,BBW,retail,<NA>,<NA>,Build-A-Bear Workshop,<NA>,<NA>,BBW,True
2,Q100321332,United Nuclear Corporation,New York Stock Exchange,UNC,mining; aviation,<NA>,<NA>,United Nuclear Corporation,<NA>,<NA>,UNC,True
3,Q100323973,Whitestone REIT,New York Stock Exchange,WSR,,<NA>,<NA>,Whitestone REIT,<NA>,<NA>,WSR,True
4,Q1007000,Genpact,New York Stock Exchange,G,professional service,<NA>,<NA>,Genpact,<NA>,<NA>,G,True
5,Q1009458,Bunge Limited,New York Stock Exchange,BG,food industry,<NA>,<NA>,Bunge Limited,67232000000.0,2022-01-01 00:00:00+00:00,BG,True
6,Q101209811,Pool Corporation,New York Stock Exchange,POOL,,<NA>,<NA>,Pool Corporation,<NA>,<NA>,POOL,True
7,Q101209850,Vontier,New York Stock Exchange,VNT,,<NA>,<NA>,Vontier,<NA>,<NA>,VNT,True
8,Q101429880,Adecoagro,New York Stock Exchange,AGRO,agribusiness; dairy industry,<NA>,<NA>,Adecoagro,<NA>,<NA>,AGRO,True
9,Q101606525,Aurora Innovation,Nasdaq,AUR,autonomous car,<NA>,<NA>,Aurora Innovation,<NA>,<NA>,AUR,True


In [8]:
df_clean.to_csv("nyse_nasdaq_companies_list.csv", index=False, encoding="utf-8")


I then Remove the Nasdaq Inc corporation manually
